In [39]:
import pandas as pd
import requests
import numpy as np
import math
import os

In [40]:
#url = http://localhost:8080/ors/v2/directions/driving-car?start=-37.81020316,145.0355016&end=-37.911108,144.993524


In [41]:
"""
ORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"
first_point = (-37.81020316, 145.0355016)
second_point = (-37.911108, 144.993524)
url = f'{ORS_BASE_URL}?start={first_point[0]},{first_point[1]}&end={second_point[0]},{second_point[1]}'

response = requests.get(url)

json_response = response.json()
json_response
"""

'\nORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"\nfirst_point = (-37.81020316, 145.0355016)\nsecond_point = (-37.911108, 144.993524)\nurl = f\'{ORS_BASE_URL}?start={first_point[0]},{first_point[1]}&end={second_point[0]},{second_point[1]}\'\n\nresponse = requests.get(url)\n\njson_response = response.json()\njson_response\n'

# Distance to primary and secondary schools


In [42]:
# read school and rental data
rental_data = pd.read_csv("../data/curated/rental_data_sf.csv")
school_data_s2 = pd.read_csv("../data/curated/school_data/school_data_s2.csv")
rental_data["coordinates"] = rental_data["coordinates"].apply(eval)
school_data_s2["coordinate"] = school_data_s2["coordinate"].apply(eval)
rental_data["SA2_CODE21"].value_counts()

SA2_CODE21
206041504.0    347
206041505.0    271
206041509.0    245
206041503.0    174
206041118.0    158
              ... 
215011391.0      1
215011393.0      1
203011034.0      1
211051283.0      1
211011257.0      1
Name: count, Length: 498, dtype: int64

In [45]:
EARTH_RADIUS = 6371
def calculate_distance(coordinate1, coordinate2):
    """ Function calculates the haversine distance between two coordinates """
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = EARTH_RADIUS * c
    
    return distance
    
    
    

In [46]:
def distance_closest_school(rental_data_coordinate, rental_data_sa2, school_data, new_col_names, school_types):
    """ Function finds the distance to the closest school from the rental property """
    
    distance_dict = {}
    # Find closest school in SA2 area
    for index, school in school_data[(school_data["sa2_code_2021"] == rental_data_sa2) 
                                     & (school_data["school_type"].isin(school_types))].iterrows():
        distance_dict[school["school_name"]] = calculate_distance(rental_data_coordinate, school["coordinate"])

    if distance_dict:
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0], 
                         new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})
    else:
        # Find closest school by comparing with all schools in victoria
        for index, school in school_data[school_data["school_type"].isin(school_types)].iterrows():
            distance_dict[school["school_name"]] = calculate_distance(rental_data_coordinate, school["coordinate"])
        if distance_dict:
            return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                             new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})
        
        
primary_school = ["primary", "pri/sec"]
new_col_names_primary = ["closest primary school", "distance primary school"]
new_columns_primary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_primary, primary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_primary], axis=1)

secondary_school = ["secondary"]
new_col_names_secondary = ["closest secondary school", "distance secondary school"]
new_columns_secondary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_secondary, secondary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_secondary], axis=1)

In [47]:
rental_data

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,NBN,Weekly Rent Amount,Postcode,geometry,SA2_CODE21,SA2_NAME21,closest primary school,distance primary school,closest secondary school,distance secondary school
0,5401/308 Exhibition Street Melbourne VIC 3000,"$3,750 weekly","[-37.8085173, 144.9696285]",Apartment / Unit / Flat,3 Beds,2 Baths,4 Parking,"BRAND NEW AT THE SAPHIRE - 3 BDR. 2.5 BTH, 4 ...",['Designed by Cox Architects and Fender Katsal...,['Built in wardrobes*'],No NBN,3750.0,3000.0,POINT (144.9696285 -37.8085173),206041503.0,Melbourne CBD - East,carlton gardens primary,0.719537,holmes secondary college,0.305406
1,4904/63 Latrobe St Melbourne VIC 3000,"$2,900/week","[-37.8082025, 144.9693999]",Apartment / Unit / Flat,3 Beds,3 Baths,2 Parking,Brand New luxurious 3 bedroom residence | Home...,['Welcome to your exclusive opportunity to res...,"['INSIDE:', 'Gym', 'Intercom', 'Floorboards', ...",No NBN,2900.0,3000.0,POINT (144.9693999 -37.8082025),206041503.0,Melbourne CBD - East,carlton gardens primary,0.684531,holmes secondary college,0.343684
2,6703/500 Elizabeth Street Melbourne VIC 3000,"$2,800","[-37.8072623, 144.9602708]",Apartment / Unit / Flat,3 Beds,3 Baths,1 Parking,3 bedroom 2.5 bathroom Penthhouse in Vision,"[""To apply, submit an enquiry and you'll recei...","['Intercom', 'Gym', 'Air conditioning', 'Study...",nbn® Fibre to the Premises (FTTP) is available...,2800.0,3000.0,POINT (144.9602708 -37.8072623),206041504.0,Melbourne CBD - North,carlton gardens primary,0.997732,ozford college,0.809262
3,4802/316-318 Queen Street Melbourne VIC 3000,$2500 Per Week,"[-37.8112667, 144.9592612]",Apartment / Unit / Flat,3 Beds,3 Baths,2 Parking,Incredibly luxurious penthouse in CBD,"[""From the moment you step inside, you'll be c...",[],No NBN,2500.0,3000.0,POINT (144.9592612 -37.8112667),206041504.0,Melbourne CBD - North,carlton gardens primary,1.364559,ozford college,0.461470
4,1201/543 Elizabeth Street Melbourne VIC 3000,"$2,500 per week","[-37.8054298, 144.9586458]",Apartment / Unit / Flat,4 Beds,4 Baths,2 Parking,Brand-new luxury Penthouse in the heart of the...,"[""This stunning addition to the north of the C...",[],No NBN,2500.0,3000.0,POINT (144.9586458 -37.8054298),206041506.0,North Melbourne,north melbourne primary,1.024508,simonds catholic college,0.509852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11803,50 Mason Street Newport VIC 3015,"$3,000 per month, plus GST & Outgoings","[-37.841798, 144.8788595]",House,4 Beds,2 Baths,4 Parking,IDEAL MEDICAL ROOMS - MAXIMUM EXPOSURE - IN TH...,['Previously operated as a medical centre but ...,"['Air conditioning*', 'Heating*']",nbn® Fibre to the Curb (FTTC) is available in ...,690.0,3015.0,POINT (144.8788595 -37.841798),213021344.0,Newport,sacred heart,0.297233,williamstown high,2.737975
11804,306 & 307/254 Hampshire Road Sunshine VIC 3020,$2700/month,"[-37.7857084, 144.8324763]",Apartment / Unit / Flat,2 Beds,2 Baths,1 Parking,READY TO MOVE INTO - IN THE HEART OF SUNSHINE,['This brand-new stylish lifestyle precinct pr...,[],nbn® Fibre to the Premises (FTTP) is available...,621.0,3020.0,POINT (144.8324763 -37.7857084),213011336.0,Sunshine,our lady of the immaculate conception,0.496611,sunshine college,1.836793
11805,25 Spring Ct Morwell VIC 3840,"$2,150.89 per month","[-38.2301648, 146.4353542]",House,3 Beds,1 Bath,2 Parking,THREE BEDROOM HOUSE,['THREE BEDROOM HOUSE- Spacious living area ad...,"['Built in wardrobes*', 'Internal Laundry*', '...",nbn® Fibre to the Node (FTTN) is available in ...,494.0,3840.0,POINT (146.4353542 -38.2301648),205041096.0,Morwell,morwell park primary,0.983446,kurnai college,0.271256
11806,133 Widford Street Glenroy VIC 3046,"$3,000 per month","[-37.6952558, 144.9248383]",House,5 Beds,1 Bath,3 Parking,2 BEDROOM BUNGALOW,['2 Bedroom BungalowLarge Front and Back YardL...,"['Air conditioning', 'Courtyard', 'Study', 'Ru...",nbn® Fibre to the Curb (FTTC) is avail

# Find closest University Campus

In [48]:
university_sa2 = pd.read_csv("../data/curated/school_data/university_location_curated_s2.csv")
university_sa2["coordinate"] = university_sa2["coordinate"].apply(eval)

In [49]:
university_sa2

,institution,campus_name,address2,suburb,state,postcode,coordinate,sa2_code_2021,sa2_name_2021
0,australian catholic university,melbourne (st patrick's),115 victoria parade,fitzroy,vic,3065,"(-37.8074982, 144.9778998)",206071142.0,Fitzroy
1,australian catholic university,melbourne (st patrick's),115 victoria parade,fitzroy,vic,3065,"(-37.8074982, 144.9778998)",206071143.0,Fitzroy North
2,australian catholic university,ballarat (aquinas),1200 mair street,ballarat,vic,3350,"(-37.5572608, 143.8436351)",201011005.0,Buninyong
3,australian catholic university,ballarat (aquinas),1200 mair street,ballarat,vic,3350,"(-37.5572608, 143.8436351)",201011484.0,Sebastopol - Redan
4,australian catholic university,ballarat (aquinas),1200 mair street,ballarat,vic,3350,"(-37.5572608, 143.8436351)",201011481.0,Ballarat East - Warrenheip
...,...,...,...,...,...,...,...,...,...
347,victoria university,werribee,hoppers lane,werribee,vic,3030,"(-37.8895758, 144.7001958)",213051368.0,Werribee - South
348,victoria university,werribee,hoppers lane,werribee,vic,3030,"(-37.8895758, 144.7001958)",213051467.0,Werribee - East
349,victoria university,werribee,hoppers lane,werribee,vic,3030,"(-37.8895758, 144.7001958)",213051466.0,Point Cook - South
350,victoria university,werribee,hoppers lane,werribee,vic,3030,"(-37.8895758, 144.7001958)",213051464.0,Point Cook - East


In [50]:
def distance_closest_university(rental_data_coordinate, rental_data_sa2, new_col_names, university_data):
    """ Function finds the distance to the closest university from the rental property """

    distance_dict = {}
    # Find closest university in SA2 area
    for index, university in university_data[university_data["sa2_code_2021"] == rental_data_sa2].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])

    for index, university in university_data[university_data["sa2_code_2021"].isna()].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])


    if distance_dict:
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                          new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})
    
    else:
        # Find closest university by comparing witha all universities in victoria
        for index, university in university_data.iterrows():
            university_campus_name = university["institution"] + "-" + university["campus_name"]
            distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])
        if distance_dict:
            return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                            new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})

In [51]:
new_col_names = ["closest university campus", "distance university campus"]
new_col_university = rental_data.apply(
    lambda x: distance_closest_university(x["coordinates"], x["SA2_CODE21"], new_col_names, university_sa2), axis=1)
rental_data = pd.concat([rental_data, new_col_university], axis=1)

In [52]:
rental_data

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,Postcode,geometry,SA2_CODE21,SA2_NAME21,closest primary school,distance primary school,closest secondary school,distance secondary school,closest university campus,distance university campus
0,5401/308 Exhibition Street Melbourne VIC 3000,"$3,750 weekly","[-37.8085173, 144.9696285]",Apartment / Unit / Flat,3 Beds,2 Baths,4 Parking,"BRAND NEW AT THE SAPHIRE - 3 BDR. 2.5 BTH, 4 ...",['Designed by Cox Architects and Fender Katsal...,['Built in wardrobes*'],...,3000.0,POINT (144.9696285 -37.8085173),206041503.0,Melbourne CBD - East,carlton gardens primary,0.719537,holmes secondary college,0.305406,southern cross university-melbourne,0.177366
1,4904/63 Latrobe St Melbourne VIC 3000,"$2,900/week","[-37.8082025, 144.9693999]",Apartment / Unit / Flat,3 Beds,3 Baths,2 Parking,Brand New luxurious 3 bedroom residence | Home...,['Welcome to your exclusive opportunity to res...,"['INSIDE:', 'Gym', 'Intercom', 'Floorboards', ...",...,3000.0,POINT (144.9693999 -37.8082025),206041503.0,Melbourne CBD - East,carlton gardens primary,0.684531,holmes secondary college,0.343684,southern cross university-melbourne,0.205431
2,6703/500 Elizabeth Street Melbourne VIC 3000,"$2,800","[-37.8072623, 144.9602708]",Apartment / Unit / Flat,3 Beds,3 Baths,1 Parking,3 bedroom 2.5 bathroom Penthhouse in Vision,"[""To apply, submit an enquiry and you'll recei...","['Intercom', 'Gym', 'Air conditioning', 'Study...",...,3000.0,POINT (144.9602708 -37.8072623),206041504.0,Melbourne CBD - North,carlton gardens primary,0.997732,ozford college,0.809262,university of the sunshine coast-usc melbourne,0.124911
3,4802/316-318 Queen Street Melbourne VIC 3000,$2500 Per Week,"[-37.8112667, 144.9592612]",Apartment / Unit / Flat,3 Beds,3 Baths,2 Parking,Incredibly luxurious penthouse in CBD,"[""From the moment you step inside, you'll be c...",[],...,3000.0,POINT (144.9592612 -37.8112667),206041504.0,Melbourne CBD - North,carlton gardens primary,1.364559,ozford college,0.461470,university of canberra-gbca - melbourne,0.028619
4,1201/543 Elizabeth Street Melbourne VIC 3000,"$2,500 per week","[-37.8054298, 144.9586458]",Apartment / Unit / Flat,4 Beds,4 Baths,2 Parking,Brand-new luxury Penthouse in the heart of the...,"[""This stunning addition to the north of the C...",[],...,3000.0,POINT (144.9586458 -37.8054298),206041506.0,North Melbourne,north melbourne primary,1.024508,simonds catholic college,0.509852,university of the sunshine coast-usc melbourne,0.318666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11803,50 Mason Street Newport VIC 3015,"$3,000 per month, plus GST & Outgoings","[-37.841798, 144.8788595]",House,4 Beds,2 Baths,4 Parking,IDEAL MEDICAL ROOMS - MAXIMUM EXPOSURE - IN TH...,['Previously operated as a medical centre but ...,"['Air conditioning*', 'Heating*']",...,3015.0,POINT (144.8788595 -37.841798),213021344.0,Newport,sacred heart,0.297233,williamstown high,2.737975,the university of melbourne-parkville,8.764619
11804,306 & 307/254 Hampshire Road Sunshine VIC 3020,$2700/month,"[-37.7857084, 144.8324763]",Apartment / Unit / Flat,2 Beds,2 Baths,1 Parking,READY TO MOVE INTO - IN THE HEART OF SUNSHINE,['This brand-new stylish lifestyle precinct pr...,[],...,3020.0,POINT (144.8324763 -37.7857084),213011336.0,Sunshine,our lady of the immaculate conception,0.496611,sunshine college,1.836793,victoria university-sunshine,0.950340
11805,25 Spring Ct Morwell VIC 3840,"$2,150.89 per month","[-38.2301648, 146.4353542]",House,3 Beds,1 Bath,2 Parking,THREE BEDROOM HOUSE,['THREE BEDROOM HOUSE- Spacious living area ad...,"['Built in wardrobes*', 'Internal Laundry*', '...",...,3840.0,POINT (146.4353542 -38.2301648),205041096.0,Morwell,morwell park primary,0.983446,kurnai college,0.271256,monash university-notting hill,119.948336
11806,133 Widford Street Glenroy VIC 3046,"$3,000 per month","[-37.6952558, 144.9248383]",House,5 Beds,1 Bath,3 

In [55]:
rental_data.to_csv("../data/curated/rental_data_distance_sf.csv", index=False)